# SPM.7(a) options → TEF transition elements — mapping

Links each AR6 SPM.7(a) mitigation option to ClimateView TEF transition elements,
in the house mapping schema (`action_to_tef_*.csv`). **Link, not attribute**:
potential (GtCO2e) stays at option level; TEs inherit a pointer, never a split value.

- Inputs: `data/spm7a_mitigation_options_2030_clean.csv` (this release) and the TEF
  catalog export `transition_elements.csv` from the `climateview-transition-elements`
  review (release 2026-02-23, in `sample/` — **gitignored, local-only**; if missing,
  re-export per that release's README).
- Output: `data/spm7a_option_to_tef.csv`
- Unmapped options are kept with `mapping_confidence: none` and a rationale.

In [1]:
import pandas as pd, re

OPTIONS = "data/spm7a_mitigation_options_2030_clean.csv"
TEF = ("../../../../climateview/climateview-transition-elements/"
       "releases/2026-02-23/sample/export-2026-02-23/transition_elements.csv")
OUT = "data/spm7a_option_to_tef.csv"

opts = pd.read_csv(OPTIONS)
tef = pd.read_csv(TEF)
tef = tef[tef.status == "active"].copy()
tef["code"] = tef.short_label.str.split(" - ").str[0].str.strip()
assert tef.code.is_unique
print(f"{len(opts)} options, {len(tef)} active TEs")

31 options, 197 active TEs


## Curated mapping

One entry per SPM.7 option. `tes` lists explicit TE codes; `pattern` selects TE
families by label regex (within a sector) where the option fans out across many
near-identical TEs — the expansion is printed below for audit. Confidence and
rationale follow the house schema; `none` means reviewed and not mappable.

In [2]:
MAPPING = [
 dict(option="Solar", tes=["T-5A1-TE-7"], confidence="high",
      rationale="Utility-scale solar electricity is the core of this option."),
 dict(option="Solar", tes=["T-5A1-TE-8"], confidence="medium",
      rationale="Rooftop solar also under SPM.7 'Onsite renewables' — overlap noted."),
 dict(option="Wind", tes=["T-5A1-TE-9", "T-5A1-TE-10"], confidence="high",
      rationale="On- and offshore wind electricity."),
 dict(option="Reduce CH4 from coal, oil and gas", tes=[], confidence="none",
      rationale="No fugitive-methane TE in catalog; not a typical municipal lever."),
 dict(option="Bioelectricity (includes BECCS)", tes=["T-5A1-TE-11", "T-5A3-TE-5"], confidence="medium",
      rationale="Biomass power and CHP; BECCS component has no TE."),
 dict(option="Geothermal and hydropower", tes=["T-5A1-TE-6"], confidence="medium",
      rationale="Hydro reservoir only; no geothermal electricity TE."),
 dict(option="Nuclear", tes=[], confidence="none",
      rationale="No nuclear TE; outside city action scope."),
 dict(option="Fossil Carbon Capture and Storage (CCS)", tes=[], confidence="none",
      rationale="No CCS TE."),
 dict(option="Reduce conversion of natural ecosystems", tes=[], confidence="none",
      rationale="No land-use-change TE; TEF AFOLU coverage is minimal."),
 dict(option="Carbon sequestration in agriculture", tes=[], confidence="none",
      rationale="No agricultural-soils TE."),
 dict(option="Ecosystem restoration, afforestation, reforestation", tes=[], confidence="none",
      rationale="No afforestation/restoration TE."),
 dict(option="Shift to sustainable healthy diets", tes=["T-3A1-TE-1"], confidence="high",
      rationale="Direct match; TE ipcc_ref 3c-02 is this option."),
 dict(option="Forest and fire management", tes=[], confidence="none",
      rationale="No forest-management TE."),
 dict(option="Reduce CH4 and N2O in agriculture", tes=[], confidence="none",
      rationale="No agricultural non-CO2 TE (T-3C6 is appliance efficiency)."),
 dict(option="Reduce food loss and food waste", tes=[], confidence="none",
      rationale="No food-waste-prevention TE; waste TEs treat disposal, not prevention."),
 dict(option="Efficient buildings", pattern=("Buildings", r"^(Retrofitting|Energy efficient new)"), confidence="high",
      rationale="Envelope/system efficiency across building-type variants (retrofit + new build)."),
 dict(option="Fuel efficient vehicles", tes=["T-1A1a-TE-4", "T-1A1c-TE-4", "T-1B1a-TE-8", "T-1B1b-TE-7"], confidence="high",
      rationale="Engine/vehicle efficiency; SYR merged light and heavy duty."),
 dict(option="Electric vehicles", tes=["T-1A1a-TE-1", "T-1A1b-TE-2", "T-1A1C-TE-5", "T-1B1a-TE-6", "T-1B1b-TE-6"], confidence="high",
      rationale="Electric road vehicles across modes; SYR merged light and heavy duty."),
 dict(option="Efficient lighting, appliances and equipment", pattern=("Buildings", r"^Energy efficient .*(lighting|electrical appliances)"), confidence="high",
      rationale="Lighting and appliance efficiency across residential/commercial/public/industrial."),
 dict(option="Public transport and bicycling", tes=["T-1A1a-TE-6", "T-1A1a-TE-10", "T-1A1a-TE-11", "T-1A1a-TE-12", "T-1A1a-TE-18"], confidence="high",
      rationale="Modal shift to public transit and active travel."),
 dict(option="Biofuels", pattern=("Transport", r"^Biofuel for"), confidence="high",
      rationale="Transport biofuel substitution across modes."),
 dict(option="Efficient shipping and aviation", tes=["T-1A3-TE-2", "T-1A4-TE-2", "T-1B3-TE-3", "T-1B4b-TE-2"], confidence="high",
      rationale="Efficiency in passenger/freight aviation and shipping."),
 dict(option="Avoid demand for energy services", tes=["T-1A1a-TE-5", "T-1A1a-TE-7", "T-1A1a-TE-8"], confidence="medium",
      rationale="Catalog covers transport demand avoidance only; option spans all sectors."),
 dict(option="Onsite renewables", tes=["T-5A1-TE-8"], confidence="medium",
      rationale="Rooftop solar (overlaps 'Solar'); plus building solar thermal below."),
 dict(option="Onsite renewables", pattern=("Buildings", r"^Shift to solar thermal"), confidence="medium",
      rationale="Building-level solar thermal as onsite renewable heat."),
 dict(option="Fuel switching", pattern=("Industry", r"^Shift to (use )?(biofuel|biogas|biomass|electricity|electric|hydrogen|inert)"), confidence="high",
      rationale="Industrial fuel/process switching to electricity, hydrogen, bio-energy."),
 dict(option="Reduce emission of fluorinated gas", tes=[], confidence="none",
      rationale="No F-gas TE."),
 dict(option="Energy efficiency", tes=["T-2D4-TE-3"], confidence="medium",
      rationale="Industrial energy efficiency only sparsely covered (mobile machinery)."),
 dict(option="Material efficiency", tes=[], confidence="none",
      rationale="No material-efficiency TE (consistent with action_to_tef_ipcc.csv ipcc_0023)."),
 dict(option="Reduce CH4 from waste/wastewater", tes=["T-6A1-TE-4"], confidence="high",
      rationale="Landfill gas recovery is the direct CH4 lever."),
 dict(option="Reduce CH4 from waste/wastewater", tes=["T-6A1-TE-2", "T-6A1-TE-3"], confidence="medium",
      rationale="Organics diversion reduces future landfill CH4."),
 dict(option="Construction materials substitution", tes=["T-4C1-TE-1", "T-4C1-TE-2", "T-4C1-TE-3"], confidence="high",
      rationale="Low-carbon construction of buildings."),
 dict(option="Enhanced recycling", tes=["T-6A1-TE-1"], confidence="high",
      rationale="Solid-waste recycling shift."),
 dict(option="Carbon capture with utilization and storage", tes=[], confidence="none",
      rationale="No CCU/CCS TE."),
]
print(f"{len(MAPPING)} curated entries for {len(set(m['option'] for m in MAPPING))} options")

34 curated entries for 31 options


## Expand patterns and resolve TE codes

In [3]:
rows = []
for m in MAPPING:
    codes = list(m.get("tes", []))
    if "pattern" in m:
        sector, rx = m["pattern"]
        fam = tef[(tef.sector_path.str.startswith(sector)) & tef.short_label.str.split(" - ").str[1].str.match(rx)]
        print(f"pattern {m['option']!r} -> {len(fam)} TEs: {fam.code.tolist()}")
        codes += fam.code.tolist()
    if not codes:
        rows.append({**m, "code": None})
    for c in codes:
        rows.append({**m, "code": c})
draft = pd.DataFrame(rows).drop(columns=["tes", "pattern"], errors="ignore")
print(f"{len(draft)} mapping rows")

pattern 'Efficient buildings' -> 12 TEs: ['T-4A1-TE-1', 'T-4A1a-TE-6', 'T-4A1a-TE-9', 'T-4A1b-TE-6', 'T-4A1b-TE-9', 'T-4B1a-TE-6', 'T-4B1a-TE-7', 'T-4B1a-TE-9', 'T-4B1b-TE-6', 'T-4B1b-TE-7', 'T-4B1c-TE-6', 'T-4B1c-TE-7']
pattern 'Efficient lighting, appliances and equipment' -> 6 TEs: ['T-4A2d-TE-1', 'T-4B2a-TE-1', 'T-4B2a-TE-2', 'T-4B2c-TE-1', 'T-4B2c-TE-2', 'T-4B2c-TE-3']
pattern 'Biofuels' -> 19 TEs: ['T-1A1a-TE-9', 'T-1A1a-TE-20', 'T-1A1a-TE-21', 'T-1A1b-TE-1', 'T-1A1c-TE-1', 'T-1A1c-TE-2', 'T-1A1c-TE-3', 'T-1A2-TE-1', 'T-1A3-TE-1', 'T-1A4-TE-3', 'T-1A4-TE-4', 'T-1B1a-TE-2', 'T-1B1a-TE-3', 'T-1B1b-TE-2', 'T-1B1b-TE-3', 'T-1B2-TE-1', 'T-1B3-TE-1', 'T-1B4b-TE-3', 'T-1B4b-TE-4']
pattern 'Onsite renewables' -> 5 TEs: ['T-4A1a-TE-4', 'T-4A1b-TE-4', 'T-4B1a-TE-5', 'T-4B1b-TE-5', 'T-4B1c-TE-5']
pattern 'Fuel switching' -> 31 TEs: ['T-2A1-TE-1', 'T-2A1-TE-2', 'T-2A3-TE-1', 'T-2A3-TE-2', 'T-2A3-TE-3', 'T-2A3-TE-4', 'T-2B9-TE-1', 'T-2B9-TE-2', 'T-2B9-TE-3', 'T-2B9-TE-4', 'T-2C1-TE-1', 'T-2C1

## Validate

In [4]:
# every option covered exactly once or more; every code resolves to an active TE
assert set(opts.option) == set(draft.option), set(opts.option) ^ set(draft.option)
mapped = draft.dropna(subset=["code"])
unknown = set(mapped.code) - set(tef.code)
assert not unknown, f"unresolvable TE codes: {unknown}"
dup = mapped.duplicated(subset=["option", "code"])
assert not dup.any(), mapped[dup]

cov = draft.groupby("option").code.apply(lambda s: s.notna().any())
print(f"options mapped to >=1 TE: {cov.sum()}/{len(cov)}")
print("unmapped:", sorted(cov[~cov].index.tolist()))
print(f"distinct TEs referenced: {mapped.code.nunique()}/{len(tef)}")

options mapped to >=1 TE: 19/31
unmapped: ['Carbon capture with utilization and storage', 'Carbon sequestration in agriculture', 'Ecosystem restoration, afforestation, reforestation', 'Forest and fire management', 'Fossil Carbon Capture and Storage (CCS)', 'Material efficiency', 'Nuclear', 'Reduce CH4 and N2O in agriculture', 'Reduce CH4 from coal, oil and gas', 'Reduce conversion of natural ecosystems', 'Reduce emission of fluorinated gas', 'Reduce food loss and food waste']
distinct TEs referenced: 110/197


## Export (house schema)

In [5]:
slug = opts.option.str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
opt_id = dict(zip(opts.option, "spm7a_" + slug))

out = mapped.merge(tef[["code", "stable_id", "short_label", "sector_path"]], on="code")
none_rows = draft[draft.code.isna()].copy()
for c in ["stable_id", "short_label", "sector_path"]:
    none_rows[c] = ""
out = pd.concat([out, none_rows], ignore_index=True)

out["src_action_id"] = out.option.map(opt_id)
out["publisher_id"] = "ipcc"
out["action_name"] = out.option
out["action_type"] = "mitigation"
out["action_role"] = "transition_element"
out = out.rename(columns={"confidence": "mapping_confidence"})
out = out[["src_action_id", "publisher_id", "action_name", "action_type", "action_role",
           "stable_id", "short_label", "sector_path", "mapping_confidence", "rationale"]]
out = out.sort_values(["src_action_id", "short_label"]).reset_index(drop=True)
out.to_csv(OUT, index=False)
print(f"wrote {OUT}: {len(out)} rows")
out.mapping_confidence.value_counts()

wrote data/spm7a_option_to_tef.csv: 123 rows


mapping_confidence
high      95
medium    16
none      12
Name: count, dtype: int64

## Findings

- **19/31 options map to >=1 TE** (123 rows: 95 high, 16 medium, 12 none); 110/197 active TEs referenced.
- **The 12 unmapped options are essentially the non-city-actionable set** — nuclear, fossil CCS, CCU, fugitive CH4, F-gases, material efficiency, food-loss prevention, and all five land/agriculture options. For city action ranking, TEF's gaps act as a relevance filter; but only 19 options' potential evidence flows through to TEs.
- Adjudicated 2026-06-05 (Amanda): all medium rows accepted as drafted, including the rooftop-solar double-link (Solar + Onsite renewables), composting as indirect waste-CH4 lever, transport-only coverage of demand avoidance, and sparse industrial energy-efficiency coverage.
- Dependency note: TEF catalog export is local-only (gitignored `sample/` in the climateview review, license verification pending); re-export instructions in that release's README.